# Gaussian Mixture Models

[k-means](kmeans.ipynb) and [DBSCAN](dbscan.ipynb) both give **hard** assignments —
each point belongs to exactly one cluster. A **Gaussian Mixture Model** makes a
genuinely different modelling assumption:

| | k-means | DBSCAN | **GMM** |
| --- | --- | --- | --- |
| assignment | hard | hard | **soft (probabilities)** |
| cluster shape | spherical | arbitrary | **elliptical (full covariance)** |
| gives probabilities? | no | no | **yes** |

GMM models the data as a mixture of Gaussians and, via Expectation-Maximization,
learns each component's **mean, covariance, and weight**. That covariance is the
key: clusters can be stretched and tilted, and a point near two components gets a
*probability* of belonging to each rather than a coin-flip. We use
`linfa-clustering`'s `GaussianMixtureModel` on data with two elongated, overlapping
clusters — exactly where k-means' spherical assumption struggles.

In [ ]:
:dep ndarray = { version = "0.15" }
:dep linfa = { version = "0.7" }
:dep linfa-clustering = { version = "0.7" }
:dep plotters = { version = "0.3", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
use ndarray::Array2;
use plotters::prelude::*;
use std::f64::consts::PI;

// Fixed-seed LCG + Box-Muller normals (top-level fns persist; no captured state).
fn next_u(seed: &mut u64) -> f64 {
    *seed = seed.wrapping_mul(6364136223846793005).wrapping_add(1);
    ((*seed >> 11) as f64) / ((1u64 << 53) as f64)
}
fn next_n(seed: &mut u64) -> f64 {
    let u1 = next_u(seed).max(1e-12);
    let u2 = next_u(seed);
    (-2.0 * u1.ln()).sqrt() * (2.0 * PI * u2).cos()
}
// 2-sigma covariance ellipse from Sigma = [[a,b],[b,c]] (principal-axis form).
fn ellipse(cx: f64, cy: f64, a: f64, b: f64, c: f64) -> Vec<(f64, f64)> {
    let tr = a + c;
    let det = a * c - b * b;
    let disc = ((tr * 0.5) * (tr * 0.5) - det).max(0.0).sqrt();
    let (l1, l2) = (tr * 0.5 + disc, (tr * 0.5 - disc).max(1e-9));
    let theta = 0.5 * (2.0 * b).atan2(a - c);
    let (sx, sy) = (2.0 * l1.sqrt(), 2.0 * l2.sqrt());
    (0..=64).map(|i| {
        let t = 2.0 * PI * i as f64 / 64.0;
        let (ex, ey) = (sx * t.cos(), sy * t.sin());
        (cx + ex * theta.cos() - ey * theta.sin(), cy + ex * theta.sin() + ey * theta.cos())
    }).collect()
}

// Two elongated Gaussian clusters (each = rotate + scale a standard normal).
let pts: Vec<(f64, f64)> = {
    let mut seed = 7u64;
    let mut out = Vec::new();
    // (center_x, center_y, angle, sx, sy)
    let specs: [(f64, f64, f64, f64, f64); 2] = [(0.0, 0.0, 0.6, 1.5, 0.35), (2.2, 1.4, -0.5, 1.5, 0.35)];
    for &(cx, cy, ang, sx, sy) in specs.iter() {
        for _ in 0..150 {
            let (z1, z2) = (next_n(&mut seed) * sx, next_n(&mut seed) * sy);
            let (rx, ry) = (z1 * ang.cos() - z2 * ang.sin(), z1 * ang.sin() + z2 * ang.cos());
            out.push((cx + rx, cy + ry));
        }
    }
    out
};
println!("{} points in 2 elongated clusters", pts.len());

## Fit GMM (and k-means, for contrast)

We fit both on the same data. From the GMM we pull out each component's **mean**
and **covariance** (for the ellipses) and compute each point's **responsibility** —
`P(cluster 1 | point)` — by hand from the Gaussian densities, since `linfa`'s GMM
exposes the parameters but not a `predict_proba`.

In [ ]:
use linfa::prelude::*;
use linfa::DatasetBase;
use linfa_clustering::{GaussianMixtureModel, KMeans};

let data: Array2<f64> = Array2::from_shape_vec(
    (pts.len(), 2), pts.iter().flat_map(|&(x, y)| [x, y]).collect()).unwrap();

// Extract everything the plot needs as plain Vecs (so evcxr persists them).
let (means_v, covs_v, weights_v, km_pred): (Vec<(f64, f64)>, Vec<(f64, f64, f64)>, Vec<f64>, Vec<usize>) = {
    let dataset = DatasetBase::from(data.clone());
    let gmm = GaussianMixtureModel::params(2)
        .n_runs(10)
        .max_n_iterations(500)
        .fit(&dataset)
        .expect("GMM fit failed");
    let m = gmm.means();
    let cv = gmm.covariances();
    let w = gmm.weights();
    let means_v = (0..2).map(|k| (m[[k, 0]], m[[k, 1]])).collect();
    let covs_v  = (0..2).map(|k| (cv[[k, 0, 0]], cv[[k, 0, 1]], cv[[k, 1, 1]])).collect();
    let weights_v = w.to_vec();
    let km = KMeans::params(2).fit(&dataset).expect("k-means fit failed");
    let km_pred = km.predict(&dataset).to_vec();
    (means_v, covs_v, weights_v, km_pred)
};

// Responsibility P(cluster 1 | point): weighted Gaussian density, normalized.
let resp1: Vec<f64> = pts.iter().map(|&(x, y)| {
    let dens = |k: usize| {
        let (mx, my) = means_v[k];
        let (a, b, c) = covs_v[k];
        let det = (a * c - b * b).max(1e-12);
        let (dx, dy) = (x - mx, y - my);
        let maha = (c * dx * dx - 2.0 * b * dx * dy + a * dy * dy) / det;
        weights_v[k] / (2.0 * PI * det.sqrt()) * (-0.5 * maha).exp()
    };
    let (p0, p1) = (dens(0), dens(1));
    p1 / (p0 + p1 + 1e-300)
}).collect();

println!("component means:   {:.2?}", means_v);
println!("component weights: {:.3?}", weights_v);
println!("points with 0.4 < P < 0.6 (uncertain): {}",
    resp1.iter().filter(|p| **p > 0.4 && **p < 0.6).count());

## k-means (hard, spherical) vs GMM (soft, elliptical)

Left: k-means assigns every point to one of two clusters with an implicitly
spherical boundary. Right: GMM — points are **coloured by responsibility** (a blend
between the two component colours, so points in the overlap go muddy/uncertain),
and the fitted **2σ covariance ellipses** are drawn, capturing the elongated tilt
that k-means cannot.

In [ ]:
let (mut xlo, mut xhi, mut ylo, mut yhi) = (f64::MAX, f64::MIN, f64::MAX, f64::MIN);
for &(x, y) in &pts { xlo = xlo.min(x); xhi = xhi.max(x); ylo = ylo.min(y); yhi = yhi.max(y); }
let pad = 0.5;
let (c0, c1) = ((30.0, 90.0, 200.0), (220.0, 120.0, 20.0)); // blue, orange

evcxr_figure((820, 380), |root| {
    root.fill(&WHITE)?;
    let panels = root.split_evenly((1, 2));
    // --- left: k-means (hard) ---
    {
        let mut ch = ChartBuilder::on(&panels[0])
            .caption("k-means (hard, spherical)", ("sans-serif", 15))
            .margin(8).x_label_area_size(28).y_label_area_size(30)
            .build_cartesian_2d((xlo - pad)..(xhi + pad), (ylo - pad)..(yhi + pad))?;
        ch.configure_mesh().disable_mesh().draw()?;
        ch.draw_series(pts.iter().enumerate().map(|(i, &(x, y))| {
            let col = if km_pred[i] == 0 { RGBColor(c0.0 as u8, c0.1 as u8, c0.2 as u8) }
                      else { RGBColor(c1.0 as u8, c1.1 as u8, c1.2 as u8) };
            Circle::new((x, y), 2, col.filled())
        }))?;
    }
    // --- right: GMM (soft) + ellipses ---
    {
        let mut ch = ChartBuilder::on(&panels[1])
            .caption("GMM (soft, elliptical)", ("sans-serif", 15))
            .margin(8).x_label_area_size(28).y_label_area_size(30)
            .build_cartesian_2d((xlo - pad)..(xhi + pad), (ylo - pad)..(yhi + pad))?;
        ch.configure_mesh().disable_mesh().draw()?;
        ch.draw_series(pts.iter().enumerate().map(|(i, &(x, y))| {
            let p = resp1[i];
            let blend = |a: f64, b: f64| ((1.0 - p) * a + p * b) as u8;
            let col = RGBColor(blend(c0.0, c1.0), blend(c0.1, c1.1), blend(c0.2, c1.2));
            Circle::new((x, y), 2, col.filled())
        }))?;
        for k in 0..2 {
            let (mx, my) = means_v[k];
            let (a, b, c) = covs_v[k];
            ch.draw_series(std::iter::once(PathElement::new(ellipse(mx, my, a, b, c), BLACK.stroke_width(2))))?;
        }
    }
    Ok(())
})

On the right, the ellipses hug the true elongated shape of each cluster, and
the points where the clusters overlap are visibly *uncertain* (blended colour) —
information k-means throws away. Where clusters really are roughly spherical and
well-separated, k-means is simpler and just as good; GMM earns its extra cost when
clusters are elongated, differently-sized, or overlapping.

## Notes & a gap

- **Soft assignments** are the point: a responsibility of 0.55 says "probably
  cluster 1, but close." Downstream you can threshold, or keep the uncertainty.
- **Choosing the number of components** is the same problem as k in k-means —
  score candidates with a criterion (BIC/AIC) or the
  [evaluation](../01d-evaluation/cross-validation.ipynb) tools rather than by eye.
- **Hierarchical / agglomerative clustering** — a common third family — has **no
  well-maintained pure-Rust crate** at the time of writing, so this guide omits it
  rather than lean on an unmaintained one (flagged in the
  [appendix](../appendix/crate-reference.md)). Re-check the ecosystem before
  assuming that's still true.

Next: back to the modelling arc with [Decision Trees](../04-trees/decision-trees.ipynb).